# Lecture 4 — Quantum Phase Transitions

---

## Overview

In lecture 02 we applied finite-size scaling to the classical Ising transition. Now we apply the exact same tools to the quantum phase transition of the 1D TFIM. The control parameter has changed — it is $\Gamma/J$ instead of temperature — but the FSS framework is identical. The quantum-to-classical mapping guarantees this: the 1D TFIM and the 2D classical Ising model share their universality class and their critical exponents.

This lecture also introduces the **spectral gap** — an observable with no classical counterpart whose closing at the quantum critical point is one of the clearest signatures of a quantum phase transition.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})


def build_tfim(N, J, Gamma):
    """Build the TFIM Hamiltonian as a sparse matrix (open boundary conditions)."""
    dim = 2**N
    H = sp.lil_matrix((dim, dim), dtype=float)
    for i in range(N - 1):
        for state in range(dim):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            H[state, state] -= J * sz_i * sz_j
    for i in range(N):
        for state in range(dim):
            H[state, state ^ (1 << i)] -= Gamma
    return H.tocsr()


def sz_op(N):
    """Total sigma_z / N operator as a sparse diagonal matrix."""
    diag = np.array([sum(1 if (s >> i) & 1 else -1 for i in range(N))
                     for s in range(2**N)], dtype=float) / N
    return sp.diags(diag, format='csr')

## 1. The Quantum Phase Transition in the TFIM

The quantum phase transition occurs at $\Gamma_c = J$:

- **Ordered phase** ($\Gamma < \Gamma_c$): ground state $\approx |\uparrow\uparrow\cdots\uparrow\rangle + |\downarrow\downarrow\cdots\downarrow\rangle$, with $\langle\hat{\sigma}^z\rangle \neq 0$ in the symmetry-broken thermodynamic limit.
- **Disordered phase** ($\Gamma > \Gamma_c$): unique ground state, $\langle\hat{\sigma}^z\rangle = 0$.
- **Critical point** ($\Gamma = J$): correlation length diverges, described by the Ising CFT with $c = 1/2$.

The role of temperature $T$ in lecture 02 is played by $\Gamma/J$ here. All FSS tools translate directly.

---

## 2. The Order Parameter

We compute the ground state $|E_0\rangle$ by exact diagonalisation and take:

$$m = \frac{1}{N}\left|\sum_i \langle E_0 | \hat{\sigma}^z_i | E_0 \rangle\right|$$

On a finite system with $\mathbb{Z}_2$ symmetry, $\langle \hat{\sigma}^z \rangle = 0$ exactly — just as in the classical case. We instead track $\langle m^2 \rangle$ and the Binder cumulant.

---

In [ ]:
# Compute order parameter observables across the phase diagram for several system sizes
sizes = [6, 8, 10, 12, 14]
gammas = np.linspace(0.2, 2.0, 60)
results = {N: {'m2': [], 'm4': [], 'gap': []} for N in sizes}

for N in sizes:
    Mz = sz_op(N)
    for g in gammas:
        H = build_tfim(N, J=1.0, Gamma=g)
        evals, evecs = eigsh(H, k=2, which='SA')
        idx = np.argsort(evals)
        psi0 = evecs[:, idx[0]]
        results[N]['gap'].append(evals[idx[1]] - evals[idx[0]])
        m2 = float((psi0 @ Mz @ Mz @ psi0).real)
        m4 = float((psi0 @ Mz @ Mz @ Mz @ Mz @ psi0).real)
        results[N]['m2'].append(m2)
        results[N]['m4'].append(m4)
    print(f"N={N} done")

print("Done.")

## 3. The Energy Gap

The spectral gap $\Delta(L, \Gamma) = E_1 - E_0$ has no classical analogue:

- **Ordered phase** ($\Gamma < \Gamma_c$): $\Delta \sim e^{-N/\xi}$ — exponentially small, reflecting quantum tunnelling between the two ordered states.
- **Disordered phase** ($\Gamma > \Gamma_c$): $\Delta \sim |\Gamma - \Gamma_c|$ — finite.
- **Critical point**: $\Delta(L, \Gamma_c) \sim L^{-z}$ with $z = 1$ for the 1D TFIM.

The dimensionless ratio $L \cdot \Delta(L, \Gamma)$ is size-independent at $\Gamma_c$ — curves for different $L$ cross at the critical point.

---

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: raw gap
for N in sizes:
    axes[0].plot(gammas, results[N]['gap'], label=f'N={N}')
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.6, label=r'$\Gamma_c=J$')
axes[0].set_xlabel(r'$\Gamma/J$')
axes[0].set_ylabel(r'$\Delta = E_1 - E_0$')
axes[0].set_title('Spectral gap')
axes[0].legend(fontsize=9)

# Right: L * gap (crossing at Gamma_c)
for N in sizes:
    axes[1].plot(gammas, N * np.array(results[N]['gap']), label=f'N={N}')
axes[1].axvline(1.0, color='red', linestyle='--', alpha=0.6, label=r'$\Gamma_c=J$')
axes[1].set_xlabel(r'$\Gamma/J$')
axes[1].set_ylabel(r'$L \cdot \Delta$')
axes[1].set_title(r'$L\cdot\Delta$ — curves cross at $\Gamma_c$')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. The Binder Cumulant for Quantum Systems

$$U_4(L, \Gamma) = 1 - \frac{\langle m^4 \rangle}{3\langle m^2 \rangle^2}$$

where moments are now ground-state expectation values. The crossing property is identical to the classical case: curves for different $L$ cross at $\Gamma_c$ without any fitting.

---

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for N in sizes:
    m2 = np.array(results[N]['m2'])
    m4 = np.array(results[N]['m4'])
    U4 = 1 - m4 / (3 * m2**2)
    ax.plot(gammas, U4, label=f'N={N}')

ax.axvline(1.0, color='red', linestyle='--', alpha=0.6, label=r'$\Gamma_c=J$')
ax.axhline(2/3, color='gray', linestyle=':', alpha=0.5, label='2/3 (ordered)')
ax.axhline(0.0, color='gray', linestyle=':', alpha=0.5, label='0 (disordered)')
ax.set_xlabel(r'$\Gamma/J$')
ax.set_ylabel(r'$U_4$')
ax.set_title('Binder cumulant — crossing at $\Gamma_c$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Finite-Size Scaling for the Gap

The gap obeys the FSS form $\Delta(L, \Gamma) = L^{-z} \cdot g\!\left((\Gamma - \Gamma_c) L^{1/\nu}\right)$.

At $\Gamma_c$, a log-log plot of $\Delta$ vs $L$ gives slope $-z$. For the 1D TFIM, $z = 1$.

---

In [ ]:
# Measure gap at the critical point for increasing N -> extract z
Ns = np.array([4, 6, 8, 10, 12, 14, 16])
gaps_at_critical = []

for N in Ns:
    H = build_tfim(N, J=1.0, Gamma=1.0)
    evals = eigsh(H, k=2, which='SA', return_eigenvectors=False)
    evals = np.sort(evals)
    gaps_at_critical.append(evals[1] - evals[0])

gaps_at_critical = np.array(gaps_at_critical)

# Fit log(Delta) = -z * log(N) + const
log_N = np.log(Ns)
log_gap = np.log(gaps_at_critical)
z_fit, const = np.polyfit(log_N, log_gap, 1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.loglog(Ns, gaps_at_critical, 'o', label='ED data')
ax.loglog(Ns, np.exp(const) * Ns**z_fit, '--', label=f'fit: slope = {z_fit:.3f} (expected -1)')
ax.set_xlabel('$N$')
ax.set_ylabel(r'$\Delta(N, \Gamma_c)$')
ax.set_title(r'Gap at $\Gamma_c$: $\Delta \sim N^{-z}$')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Extracted dynamical exponent z = {-z_fit:.4f} (expected z = 1)")

## 6. Scaling Collapse for the Quantum Transition

With $\tilde{m} = L^{\beta/\nu} \langle |\hat{m}^z| \rangle$ and $x = (\Gamma - \Gamma_c) \cdot L^{1/\nu}$, all system sizes should collapse onto a single curve using $\beta = 1/8$ and $\nu = 1$ — the same exponents as the 2D classical Ising model.

---

In [ ]:
# Scaling collapse
Gamma_c = 1.0
beta_over_nu = 1/8
nu = 1.0

fig, ax = plt.subplots(figsize=(7, 5))

for N in sizes:
    m2 = np.array(results[N]['m2'])
    m_abs = np.sqrt(m2)  # approximate |m| from <m^2>^{1/2}
    x = (gammas - Gamma_c) * N**(1/nu)
    y = N**beta_over_nu * m_abs
    ax.plot(x, y, label=f'N={N}')

ax.set_xlabel(r'$(\Gamma - \Gamma_c) \cdot L^{1/\nu}$')
ax.set_ylabel(r'$L^{\beta/\nu} \langle|m^z|\rangle$')
ax.set_title(r'Scaling collapse ($\beta/\nu=1/8$, $\nu=1$)')
ax.set_xlim(-8, 8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Comparison: Classical vs Quantum Critical Behaviour

| Observable | Classical Ising (lecture 02) | Quantum TFIM (lecture 04) |
|---|---|---|
| Control parameter | Temperature $T$ | Transverse field $\Gamma/J$ |
| Order parameter | $\langle\|m\|\rangle$ (thermal) | $\langle\|\hat m^z\|\rangle$ (ground state) |
| Gap closes? | No — thermal gap always finite | Yes — spectral gap $\Delta \sim L^{-z}$ |
| Binder cumulant | Crosses at $T_c$ | Crosses at $\Gamma_c$ |
| Critical exponents | $\beta=1/8$, $\nu=1$, $\gamma=7/4$ | Same (universality class) |

The diverging correlation length and the vanishing spectral gap are two faces of the same phenomenon: the system is poised between two competing phases, unable to pick either.

---